In [1]:
setwd("/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb")
library(data.table)
library(reshape2)


Attaching package: 'reshape2'


The following objects are masked from 'package:data.table':

    dcast, melt




In [14]:
path <- "data/permute/genes/chr7/ukb_eur_wes_200k_pLoF_damaging_missense_chr7_ENSG00000164692.tsv.gz"

In [15]:
d <- fread(path, tmpdir = "data/tmp/rtmp/")

In [16]:
shuffle_knockouts <- function(d){
    d$KO <- rbinom(n=nrow(d), size=1, prob = d$pTKO)
    d$pKO <- ifelse((d$KO == 1), 1,
             ifelse((d$phased_het == 1 & d$unphased_het > 0), 1 - 1*(1/2)^d$unphased_het,
             ifelse((d$phased_het == 0 & d$unphased_het > 1), 1 - 2*(1/2)^d$unphased_het, 0)))
    return(d$pKO)
}

In [17]:
n = 10

In [19]:
# where is 3717495 (0.5)
# where is 1365568 (0.5)
# where is 1466583 (0.5)
weird_samples <- c(3717495, 1365568, 1466583)

In [20]:
d[d$s %in% c(weird_samples)]

gene_id,s,unphased_het,phased_het,hom_alt_n,pTKO
<chr>,<int>,<int>,<int>,<int>,<dbl>
ENSG00000164692,1365568,1,1,0,0
ENSG00000164692,1466583,2,0,0,0
ENSG00000164692,3717495,1,1,0,0


In [ ]:
#ENSG00000164692	6026401	0	0	0	0.0000e+00
#ENSG00000164692	1326715	0	2	0	5.0000e-01
#ENSG00000164692	1890022	0	2	0	5.0000e-01
#ENSG00000164692	2301892	0	2	0	5.0000e-01
#ENSG00000164692	2314440	0	2	0	5.0000e-01
#ENSG00000164692	5571743	0	2	0	5.0000e-01

# where is 3717495 (0.5)
# where is 1365568 (0.5)
# where is 1466583 (0.5)

In [50]:
reps <- replicate(n, shuffle_knockouts(d))
rownames(reps) <- d$s
reps <- data.table(t(reps))

In [57]:
mat <- reps[,colSums(reps) > 0, with = FALSE]
mat <- mat[rowSums(mat) == 5, ]
ok <- mat[["1326715"]] == 1 & 
      #mat[["1890022"]] == 0 & 
      #mat[["2301892"]] == 0 & 
      mat[["2314440"]] == 0 
mat[ok, ]

#melted_mat <- melt(mat)

1326715,1365568,1466583,1702288,1890022,2301892,2314440,3717495,5571743
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,0.5,0.5,0.5,0,1,0,0.5,1
1,0.5,0.5,0.5,1,1,0,0.5,0
1,0.5,0.5,0.5,1,1,0,0.5,0
1,0.5,0.5,0.5,1,0,0,0.5,1
1,0.5,0.5,0.5,1,0,0,0.5,1
1,0.5,0.5,0.5,1,1,0,0.5,0
1,0.5,0.5,0.5,1,1,0,0.5,0
1,0.5,0.5,0.5,1,0,0,0.5,1
1,0.5,0.5,0.5,1,1,0,0.5,0


In [52]:
#mat

1326715,1365568,1466583,1702288,1890022,2301892,2314440,3717495,5571743
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
0,0.5,0.5,0.5,0,1,1,0.5,1
0,0.5,0.5,0.5,1,1,0,0.5,1
0,0.5,0.5,0.5,1,1,0,0.5,1
1,0.5,0.5,0.5,1,0,1,0.5,0
0,0.5,0.5,0.5,0,1,1,0.5,1
0,0.5,0.5,0.5,1,0,1,0.5,1
0,0.5,0.5,0.5,1,1,0,0.5,1
1,0.5,0.5,0.5,0,1,0,0.5,1
0,0.5,0.5,0.5,0,1,1,0.5,1


In [9]:
p99 <- "data/permute/permutations/chr7/ENSG00000164692/ukb_eur_wes_200k_pLoF_damaging_missense_permuted_chr7_ENSG00000164692_99.vcf.gz"
p98 <- "data/permute/permutations/chr7/ENSG00000164692/ukb_eur_wes_200k_pLoF_damaging_missense_permuted_chr7_ENSG00000164692_98.vcf.gz"

In [10]:
d99 <- fread(p99, tmpdir = "data/tmp/rtmp/", skip = '##', nrows = 10)
d98 <- fread(p98, tmpdir = "data/tmp/rtmp/", skip = '##', nrows = 10)

In [13]:
d99[1,]
d98[1,]

#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,1000028,⋯,6026237,6026268,6026270,6026295,6026311,6026322,6026335,6026383,6026397,6026401
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
chr7,1,ENSG00000164692,X,Y,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0


#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,1000028,⋯,6026237,6026268,6026270,6026295,6026311,6026322,6026335,6026383,6026397,6026401
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
chr7,1,ENSG00000164692,X,Y,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0


In [89]:
head(d, 2)

#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,1000028,⋯,6026237,6026268,6026270,6026295,6026311,6026322,6026335,6026383,6026397,6026401
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
chr7,1,ENSG00000164692,X,Y,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0
chr7,2,ENSG00000164692,X,Y,.,.,.,DS,0,⋯,0,0,0,0,0,0,0,0,0,0


In [83]:
table(d$p.value)


0.653154 0.707262 0.723476 0.725199  0.80211 0.894845 0.903404 0.913869 
   62506    62443    62469    62624    62550    62412    62833    62593 
0.922451 0.968206 0.975673 
   62163    62491    62413 

In [77]:
fread()

V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,⋯,V161275,V161276,V161277,V161278,V161279,V161280,V161281,V161282,V161283,V161284
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
#CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,FORMAT,1000028,⋯,5584149,5584165,5584201,5584225,5584299,5584312,5584377,5584402,5584416,55
